In [62]:
from data_setup import *
import polars as pl

In [64]:
dl = pl.read_database_uri('select * from pd023',connMy)
dl

月份,高手
i64,str
1,"""阳顶天"""
1,"""狄云"""
1,"""虚竹, 杨过"""
1,"""独孤求败, 风清扬"""
1,"""段誉"""
1,"""小龙女"""
1,"""慕容博"""
1,"""瑛姑，黄蓉"""
2,"""风清扬"""


In [96]:
dl.group_by(
    pl.first(),maintain_order=True
).agg(pl.last().str.join(",")).select(
    pl.first(),
    pl.concat_str(
        pl.last(),pl.last().shift(-1),pl.last().shift(-2),separator=","
    )
).drop_nulls().with_columns(
    pl.last().str.extract_all(r'[\u4e00-\u9fff]+').list.eval(
        pl.element().value_counts(sort=True)
    )
).explode(pl.last()).unnest(pl.last()).filter(
    pl.last() == pl.last().max().over(pl.first())
).group_by(pl.nth(1),maintain_order=True).agg(
    pl.len().alias("count")
).sort(pl.col("count"),descending=True).head(3)

,count
str,u32
"""张三丰""",3
"""段誉""",3
"""火云邪神""",3


In [104]:
(
    dl.group_by(pl.first(), maintain_order=True)
    .agg(pl.last().str.join(","))
    .with_columns(
        pl.concat_str(pl.last(), pl.last().shift(-1), pl.last().shift(-2), separator=",")
        .str.extract_all(r'[\u4e00-\u9fff]+')
        .list.eval(
        pl.element().value_counts(sort=True)
    )
    ))

月份,高手
i64,list[struct[2]]
1,"[{""风清扬"",3}, {""黄蓉"",3}, … {""扫地僧"",1}]"
2,"[{""乔峰"",3}, {""张三丰"",3}, … {""虚竹"",1}]"
3,"[{""郭靖"",3}, {""扫地僧"",3}, … {""张无忌"",1}]"
4,"[{""阳顶天"",3}, {""张三丰"",3}, … {""紫衫龙王"",1}]"
5,"[{""袁承志"",2}, {""瑛姑"",2}, … {""灭绝师太"",1}]"
6,"[{""段誉"",3}, {""石破天"",3}, … {""令狐冲"",1}]"
7,"[{""段誉"",3}, {""火云邪神"",3}, … {""杨过"",1}]"
8,"[{""令狐冲"",3}, {""袁承志"",2}, … {""紫衫龙王"",1}]"
9,null


In [ ]:
(
    dl
    .group_by(pl.first(), maintain_order=True)
    .agg(pl.last())
    .with_columns(
        pl.concat_list(
            [pl.last().shift(-i) for i in range(3)]
        )
    ).drop_nulls(pl.last()).with_columns(
        pl.last().list.eval(
            pl.element().str.extract_all(r'[\u4e00-\u9fff]+')
        )
    ).explode(pl.last()).explode(pl.last())
)

月份,高手
i64,str
1,"""阳顶天"""
1,"""狄云"""
1,"""虚竹"""
1,"""杨过"""
1,"""独孤求败"""
1,"""风清扬"""
1,"""段誉"""
1,"""小龙女"""
1,"""慕容博"""


In [107]:
expr = pl.col("abc")

expr.meta.output_name()

'abc'

In [103]:
pl.concat_str(
    [pl.col("combined").shift(-i) for i in range(3)],
    separator=","
).alias("window")

<Expr ['col("combined").shift([dyn int…'] at 0x1D76F45A050>